## Metatranscriptomics processing

In [6]:
import qiime2
import biom
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
import os

%matplotlib inline
import glob

## get the depth of the genome.tsv file for the not norm table

max depth of mtg 113,935

In [ ]:
biom convert \
-i /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.tsv \
-o /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.biom \
--table-type="OTU table" \
--to-hdf5

qiime tools import \
--input-path /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.biom \
--output-path /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.qza \
--type FeatureTable[Frequency]

mtx genome depth is 60,610,350

In [ ]:
qiime diversity alpha-rarefaction \
  --i-table /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.qza \
  --p-max-depth 80000000 \
  --p-min-depth 500000 \
  --p-steps 10 \
  --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/metaT_metadata_ztcat_noNT.txt \
  --output-dir /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-alpha-rarefaction80M \
  --o-visualization /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-alpha-rarefaction80M.qzv

get rarefied table that matches depth of MGX

In [ ]:
qiime diversity core-metrics \
  --i-table /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/genome_clean_noNT.qza \
  --p-sampling-depth 113935 \
  --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/metaT_metadata_ztcat_noNT.txt \
  --output-dir /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-diversity-core-metrics114K/

In [ ]:
qiime tools export --input-path /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-diversity-core-metrics114K/rarefied_table.qza \
    --output-path /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-diversity-core-metrics114K/rarefied_table

In [ ]:
biom convert -i /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-diversity-core-metrics114K/rarefied_table/feature-table.biom \
    -o /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/g-diversity-core-metrics114K/rarefied_table/feature-table.tsv \
    --to-tsv

In [ ]:
woltka tools collapse -i feature-table.tsv -m /mnt/zarrinpar/Pynchon/Databases/wol2/function/pfam/orf-to-pfam.map.xz -n /mnt/zarrinpar/Pynchon/Databases/wol2/function/pfam/pfam_description.txt -o pfam-144k.tsv

In [ ]:
woltka normalize -i feature-table.tsv --sizes /mnt/zarrinpar/Pynchon/Databases/wol2/proteins/coords.txt.xz --scale 1k --digits 3 -o feature-table-rpk.tsv

In [ ]:
woltka normalize -i feature-table-rpk.tsv --scale 1M -o feature-table-TPM.tsv

In [ ]:
woltka tools collapse -i feature-table-TPM.tsv -m /mnt/zarrinpar/Pynchon/Databases/wol2/function/pfam/orf-to-pfam.map.xz -n /mnt/zarrinpar/Pynchon/Databases/wol2/function/pfam/pfam_description.txt -o pfam-144k_TPM.tsv

## Get beta diversity

In [ ]:
qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_clean_noNT.qza \
    --p-min-feature-count 15 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/ordination_deicode_TRFmetaT_noNT.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT.qza

In [ ]:
qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/ordination_deicode_TRFmetaT_noNT.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/metaT_metadata_noNT.txt \
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/ordination_deicode_TRFmetaT_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/metaT_metadata_noNT.txt \
    --m-metadata-column condition \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT-significanceCond.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/pfam/rpca_results/distance_matrix.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/metaT_metadata_ztcat_noNT.txt \
    --m-metadata-column cond_phase \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/pfam/rpca_results/condphase-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/pfam/rpca_results/distance_matrix.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/metaT_metadata_ztcat_noNT.txt \
    --m-metadata-column cond_zt \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/TRF_multiomics/metatranscript/woltka2_m_results/pfam/rpca_results/condzt-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/metaT_metadata_noNT.txt \
    --m-metadata-column lightdark \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT-significanceLD.qzv

In [ ]:
qiime tools export \
    --input-path /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT.qza \
    --output-path /mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/beta_deicode_TRFmetaT_noNT

**convert counts tables to TPM**

In [13]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FAFT.txt',
                     sep='\t', index_col=0)
df.head()

,cFA01a,cFA01b,cFA05a,cFA05b,cFA09a,cFA09b,cFA13a,cFA13b,cFA17a,cFA17b,...,cFT05a,cFT05b,cFT09a,cFT09b,cFT13a,cFT13b,cFT17a,cFT17b,cFT21a,cFT21b
FeatureID,,,,,,,,,,,,,,,,,,,,,
PF13561.9,1227.444,661.210,871.082,705.518,1279.808,874.213,844.390,1336.959,1135.773,1055.530,...,767.759,1119.805,1208.059,845.434,945.915,588.511,991.704,747.552,471.290,557.205
PF08240.15,2329.862,776.307,954.344,571.605,997.675,1248.008,932.635,924.760,878.857,917.060,...,1024.555,1054.666,823.046,957.627,1118.048,1185.505,1076.582,1093.462,692.260,715.475
PF00107.29,2320.324,771.913,942.132,568.493,979.348,1250.633,925.556,910.447,875.142,908.918,...,1025.110,1061.912,810.929,959.374,1108.410,1185.545,1076.215,1086.574,694.061,716.446
PF12698.10,256.816,70.840,429.462,84.284,329.369,97.042,176.926,348.855,144.468,141.826,...,149.060,413.947,256.015,146.941,391.488,171.223,480.765,250.287,71.600,60.126
PF01264.24,98.301,71.636,119.419,46.923,88.573,95.642,74.473,129.736,74.204,109.146,...,21.647,56.821,108.724,53.731,61.392,53.505,56.753,57.604,55.616,38.639


In [14]:
df = df * 1000000 / df.sum()
df.head()

,cFA01a,cFA01b,cFA05a,cFA05b,cFA09a,cFA09b,cFA13a,cFA13b,cFA17a,cFA17b,...,cFT05a,cFT05b,cFT09a,cFT09b,cFT13a,cFT13b,cFT17a,cFT17b,cFT21a,cFT21b
FeatureID,,,,,,,,,,,,,,,,,,,,,
PF13561.9,290.152308,318.938010,118.945677,469.137006,411.935037,314.135808,263.543486,486.066428,382.549286,392.022535,...,356.400190,534.366809,210.596896,307.929526,194.514677,242.480643,148.536793,362.926854,165.340359,222.631694
PF08240.15,550.750044,374.455634,130.315049,380.091023,321.124175,448.453639,291.085729,336.206862,296.015240,340.594948,...,475.607055,503.282719,143.478864,348.793197,229.911509,488.456486,161.249766,530.861698,242.862180,285.868596
PF00107.29,548.495380,372.336166,128.647508,378.021686,315.225217,449.396895,288.876295,331.003210,294.763960,337.571020,...,475.864691,506.740483,141.366547,349.429501,227.929585,488.472966,161.194797,527.517662,243.494016,286.256560
PF12698.10,60.708069,34.170035,58.642755,56.044982,106.014833,34.870640,55.220567,126.830145,48.659486,52.674001,...,69.194907,197.533979,44.630241,53.519816,80.504233,70.547982,72.008675,121.511110,25.119077,24.023390
PF01264.24,23.237119,34.553989,16.306586,31.201636,28.509215,34.367571,23.243849,47.166977,24.993275,40.536689,...,10.048720,27.114771,18.953492,19.570258,12.624438,22.045343,8.500428,27.965999,19.511488,15.438243


In [15]:
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FAFT_TPM.txt',
         sep='\t')

In [16]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FTNA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FTNA_TPM.txt',
         sep='\t')

In [17]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FANA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_FANA_TPM.txt',
         sep='\t')

In [18]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_clean_noNT.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/pfam/pfam_clean_noNT_TPM.txt',
         sep='\t')

In [2]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/species_func/species_pfam_clean_noNT.tsv',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/species_func/species_pfam_clean_noNT_TPM.txt',
         sep='\t')

In [2]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/genus_func/genus_pfam_clean_noNT.tsv',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/genus_func/genus_pfam_clean_noNT_TPM.txt',
         sep='\t')

In [10]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_FA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_FA_TPM.txt',
         sep='\t')

In [11]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_NA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_NA_TPM.txt',
         sep='\t')

In [12]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_FT.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_SFR/filtered_rna_FT_TPM.txt',
         sep='\t')

In [7]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_FA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_FA_TPM.txt',
         sep='\t')

In [8]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_NA.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_NA_TPM.txt',
         sep='\t')

In [9]:
df= pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_FT.txt',
                     sep='\t', index_col=0)
df = df * 1000000 / df.sum()
df.to_csv('/mnt/zarrinpar/scratch/sfloresr/metatranscript/metatranscript/woltka2_results/aidan_wol2_cyclic_analysis/wol2_rna_pipeline_144K_SFR/filtered_rna_FT_TPM.txt',
         sep='\t')